In [2]:
import polars as pl
pl.Config.set_tbl_rows(30)

polars.config.Config

In [3]:
df = pl.read_csv("results.csv")

In [4]:
(
    df
    .filter(pl.col("split").eq("val"))
    .group_by("method", "model_version", "reasoning", "effort")
    .agg(
        pl.col("correct_square_mean").mean(),
        pl.col("correct_board_mean").mean(),
        pl.len().alias("n")
    )
    .sort("correct_board_mean", descending=True)
    .filter(pl.col("method").eq("cnn"))
)

method,model_version,reasoning,effort,correct_square_mean,correct_board_mean,n
str,str,str,str,f64,f64,u32
"""cnn""","""optimised_plus_prior_correctio…","""none""","""high""",0.739553,0.947831,5
"""cnn""","""optimised""","""none""","""high""",0.727543,0.910303,5
"""cnn""","""optimised_10k""","""none""","""high""",0.708313,0.909649,5
"""cnn""","""square_global""","""none""","""high""",0.716104,0.859952,5
"""cnn""","""square_per_square""","""none""","""high""",0.759768,0.858538,5
"""cnn""","""optimised_5k""","""none""","""high""",0.63859,0.788188,5
"""cnn""","""none_global""","""none""","""high""",0.602451,0.307201,5


In [5]:
(
    df.filter(
        pl.col("model_version").str.starts_with("optimised"), 
        ~pl.col("model_version").str.contains("prior"),
        pl.col("split").eq("val")
    )
    .group_by("method", "model_version")
    .agg(
        pl.col("correct_square_mean").mean(),
        pl.col("correct_board_mean").mean()
    )
    .select(
        pl.col("method").replace_strict({"cnn": "CNN"}).alias("model type"),
        pl.col("model_version").replace_strict({"optimised": 13184, "optimised_10k": 9538, "optimised_5k": 4864}).alias("n"),
        pl.col("correct_square_mean").round(3).alias("square accuracy"),
        pl.col("correct_board_mean").round(3).alias("move accuracy")
    )
).write_csv("results_data_ablation.csv")

In [6]:
df.filter(pl.col("method").eq("cnn")).unique(["model_version"])

method,model_version,prompt_version,reasoning,prior_correction,data_path,split,setup_id,n_boards,correct_square,correct_square_mean,correct_board,correct_board_mean,board_rank,board_rank_mean,first_output_illegal,first_output_illegal_mean,none_legal,none_legal_mean,input_tokens,input_tokens_mean,output_tokens,output_tokens_mean,inference_time,inference_time_mean,cost,cost_mean,n_suggested,n_suggested_mean,effort
str,str,i64,str,bool,str,str,str,i64,str,f64,str,f64,str,f64,str,str,str,str,str,f64,str,f64,str,f64,str,f64,str,str,str
"""cnn""","""optimised_5k""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-09_205328""",18,"""[0.625, 0.625, 0.53125, 0.625,…",0.587674,"""[true, true, true, true, true,…",0.888889,"""[1.0, 1.0, 1.0, 1.0, 1.0, 1.0,…",0.934965,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[1.7189999999973224, 2.5779999…",2.139833,"""[0.0, 0.0, 0.0, 0.0, 0.0, 0.0,…",0.0,null,null,"""high"""
"""cnn""","""none_global""",1,"""none""",false,"""data/generated_none_global/dat…","""val""","""2026-07-09_205328""",18,"""[0.578125, 0.5625, 0.5625, 0.5…",0.5859375,"""[true, true, false, true, fals…",0.277778,"""[1.0, 1.0, 0.38095238095238093…",0.871688,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[4.281000000000006, 4.29699999…",7.210111,"""[0.0, 0.0, 0.0, 0.0, 0.0, 0.0,…",0.0,null,null,"""high"""
"""cnn""","""optimised_plus_prior_correctio…",1,"""none""",true,"""data/generated/data.csv""","""val""","""2026-07-09_205328""",18,"""[0.65625, 0.671875, 0.46875, 0…",0.657986,"""[true, true, true, true, true,…",0.944444,"""[1.0, 1.0, 1.0, 1.0, 1.0, 1.0,…",0.993827,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[2.7339999999967404, 2.7350000…",2.848,"""[0.0, 0.0, 0.0, 0.0, 0.0, 0.0,…",0.0,null,null,"""high"""
"""cnn""","""square_global""",1,"""none""",false,"""data/generated_square_global/d…","""val""","""2026-07-09_205328""",18,"""[0.65625, 0.640625, 0.578125, …",0.65625,"""[true, false, true, true, true…",0.777778,"""[1.0, 0.95, 1.0, 1.0, 1.0, 1.0…",0.972884,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[4.859000000000037, 4.51599999…",7.2135,"""[0.0, 0.0, 0.0, 0.0, 0.0, 0.0,…",0.0,null,null,"""high"""
"""cnn""","""optimised_10k""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-09_205328""",18,"""[0.671875, 0.640625, 0.578125,…",0.652778,"""[true, true, true, true, true,…",0.944444,"""[1.0, 1.0, 1.0, 1.0, 1.0, 1.0,…",0.989198,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[2.3289999999979045, 2.3600000…",2.373333,"""[0.0, 0.0, 0.0, 0.0, 0.0, 0.0,…",0.0,null,null,"""high"""
"""cnn""","""square_per_square""",1,"""none""",false,"""data/generated_square_per_squa…","""val""","""2026-07-09_205328""",18,"""[0.71875, 0.734375, 0.625, 0.7…",0.743924,"""[true, true, true, true, true,…",0.888889,"""[1.0, 1.0, 1.0, 1.0, 1.0, 1.0,…",0.975529,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…",0.0,"""[3.812999999999988, 4.375, 4.3…",4.23,"""[0.0, 0.0, 0.0, 0.0, 0.0, 0.0,…",0.0,null,null,"""high"""
"""cnn""","""optimised""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-09_205328""",18,"""[0.765625, 0.765625, 0.671875,…",0.703993,"""[true, true, true, true, true,…",0.944444,"""[1.0, 1.0, 1.0, 1.0, 1.0, 1.0,…",0.996914,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",

In [7]:
df.filter(pl.col("model_version").eq("none_global"), pl.col("split").eq("val"))["correct_board_mean"].mean()

0.3072014260249555

In [8]:
# Evaluate effectiveness of mask;
# Baseline: none_global;
# square_global
# optimised
(
    df.filter(
        pl.col("model_version").is_in(["none_global", "square_global", "optimised"]),
        pl.col("split").eq("val")
    )
    .group_by("method", "model_version")
    .agg(
        pl.col("correct_square_mean").mean(),
        pl.col("correct_board_mean").mean()
    )
    .select(
        pl.col("method").replace_strict({"cnn": "CNN"}).alias("model type"),
        pl.col("model_version").replace_strict({"none_global": "no masks, global padding", "square_global": "square mask, global padding", "optimised": "full mask, square-dependent padding"}),
        pl.col("correct_square_mean").round(3).alias("square accuracy"),
        pl.col("correct_board_mean").round(3).alias("move accuracy")
    )
).write_csv("results_geometric_ablation.csv")

In [9]:
model_version_map = {
    "claude-opus-5": "Opus 5",
    "claude-sonnet-5": "Sonnet 5"
}

reasoning_enabled_map = {
    "thinking": "yes",
    "none": "no"
}




(
    df
    .filter(pl.col("split").eq("val"))
    .group_by("method", "model_version", "reasoning", "effort")
    .agg(
        pl.col("correct_square_mean").mean(),
        pl.col("correct_board_mean").mean(),
        pl.len().alias("n")
    )
    .sort("correct_board_mean", descending=True)
    .group_by("method", "model_version", "reasoning")
    .agg(
        pl.col("correct_square_mean").first(),
        pl.col("correct_board_mean").first(),
        pl.col("effort").first()
    )
    .filter(~pl.col("method").is_in(["cnn", "fen_whole"]), pl.col("model_version").ne("claude-opus-4-8"))
    .select(
        pl.col("model_version").replace_strict(model_version_map).alias("model version"),
        pl.col("method").str.starts_with("square").replace_strict({True: "yes", False: "no"}).alias("classify squares first?"),
        pl.col("method").alias("prompt version"),
        pl.when(pl.col("reasoning").eq("thinking"))
        .then(pl.col("effort").replace_strict({"low": "yes (low effort)", "medium": "yes (medium effort)", "high": "yes (high effort)"}))
        .otherwise(pl.lit("no"))
        .alias("reasoning enabled?"),
        pl.col("correct_board_mean").alias("move accuracy"),
        pl.col("correct_square_mean").alias("square accuracy")
    )
    .with_columns(
        pl.when(pl.col("classify squares first?").eq("no"))
        .then(pl.lit(None))
        .otherwise(pl.col("square accuracy"))
        .alias("square accuracy")
    )
).write_csv("claude_results.csv")

# How to turn this into an actual table I can show in the blog post?
# Columns need to be renamed 
    # reasoning -> Reasoning enabled?
    # correct_square_mean -> Square Accuracy
    # correct_board_mean -> Move Accuracy
# Values need to be renamed:
    # claude-opus-5 -> Opus 5
    # correct

In [10]:
df.filter(pl.col("model_version").eq("claude-opus-5"), pl.col("split").eq("test"))

method,model_version,prompt_version,reasoning,prior_correction,data_path,split,setup_id,n_boards,correct_square,correct_square_mean,correct_board,correct_board_mean,board_rank,board_rank_mean,first_output_illegal,first_output_illegal_mean,none_legal,none_legal_mean,input_tokens,input_tokens_mean,output_tokens,output_tokens_mean,inference_time,inference_time_mean,cost,cost_mean,n_suggested,n_suggested_mean,effort
str,str,i64,str,bool,str,str,str,i64,str,f64,str,f64,str,f64,str,str,str,str,str,f64,str,f64,str,f64,str,f64,str,str,str
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-01_210216""",7,"""[0.484375, 0.4375, 0.46875, 0.…",0.448661,"""[true, true, false, false, fal…",0.285714,"""[1.0, 1.0, 0.43333333333333335…",0.874044,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[55577, 55577, 55577, 55577, 5…",55577.0,"""[5028, 5080, 5099, 5161, 5084,…",5099.857143,"""[null, null, null, null, null,…",null,"""[0.20179250000000004, 0.202442…",0.202691,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-06_204708""",9,"""[0.421875, 0.5, 0.5625, 0.4062…",0.458333,"""[false, false, false, true, fa…",0.111111,"""[0.5588235294117647, 0.875, 0.…",0.873473,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[54902, 54902, 54902, 54902, 5…",54902.0,"""[4912, 5222, 5002, 5208, 5155,…",5240.111111,"""[null, null, null, null, null,…",null,"""[0.19865500000000003, 0.202530…",0.202756,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-07_083055""",8,"""[0.578125, 0.5625, 0.515625, 0…",0.542969,"""[true, false, false, false, fa…",0.375,"""[1.0, 0.9411764705882353, 0.97…",0.910561,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[55013, 55013, 55013, 55013, 5…",55013.0,"""[4957, 4969, 5166, 5042, 4967,…",4987.5,"""[null, null, null, null, null,…",null,"""[0.199495, 0.19964500000000002…",0.199876,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-07_090013""",7,"""[0.578125, 0.4375, 0.4375, 0.4…",0.439732,"""[true, false, false, false, fa…",0.142857,"""[1.0, 0.875, 0.928571428571428…",0.856468,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[54939, 54939, 54939, 54939, 5…",54939.0,"""[5135, 5301, 5092, 5640, 5122,…",5272.0,"""[null, null, null, null, null,…",null,"""[0.20153500000000002, 0.20361,…",0.2032475,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-07_090457""",6,"""[0.390625, 0.453125, 0.5, 0.48…",0.46875,"""[false, false, false, false, f…",0.0,"""[0.972972972972973, 0.75, 0.53…",0.769463,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[55287, 55287, 55287, 55287, 5…",55287.0,"""[5626, 5039, 5256, 5186, 4907,…",5185.666667,"""[null, null, null, null, null,…",null,"""[0.20854250000000002, 0.201205…",0.203038,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-opus-5""",1,"""thinking""",false,"""data/generated/data.csv""","""test""","""2026-07-07_090950""",7,"""[0.546875, 0.46875, 0.53125, 0…",0.495536,"""[false, false, false, false, f…",0.0,"""[0.8409090909090909, 0.9555555…",0.791897,"""[null, null, null, null, null,…",null,"""[null, null, null, null, null,…",null,"""[55599, 55599, 55599, 55599, 5…",55599.0,"""[4864, 4959, 5068, 5140, 4914,…",5009.571429,"""[null, null, null, null, null,…",null,"""[0.19979750000000002, 0.200985…",0.201617,"""[null, null, null, null, null,…",null,"""low"""
"""square_logits""","""claude-

In [11]:
import os
os.getcwd()

'c:\\Users\\User\\Documents\\Coding\\robot-chess-commentator\\evaluation'

In [12]:
df.filter(pl.col("model_version").eq("claude-sonnet-5"), pl.col("method").eq("board")).select("reasoning", "correct_square", "correct_square_mean", "correct_board", "correct_board_mean", "board_rank", "n_suggested", "none_legal", "first_output_illegal", "n_suggested_mean"),

(shape: (10, 10)
 ┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
 │ reasoning ┆ correct_s ┆ correct_s ┆ correct_b ┆ … ┆ n_suggest ┆ none_lega ┆ first_out ┆ n_sugges │
 │ ---       ┆ quare     ┆ quare_mea ┆ oard      ┆   ┆ ed        ┆ l         ┆ put_illeg ┆ ted_mean │
 │ str       ┆ ---       ┆ n         ┆ ---       ┆   ┆ ---       ┆ ---       ┆ al        ┆ ---      │
 │           ┆ str       ┆ ---       ┆ str       ┆   ┆ str       ┆ str       ┆ ---       ┆ str      │
 │           ┆           ┆ f64       ┆           ┆   ┆           ┆           ┆ str       ┆          │
 ╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
 │ none      ┆ [0.9375,  ┆ 0.944444  ┆ [false,   ┆ … ┆ [1, 1, 1, ┆ [false,   ┆ [false,   ┆ 1.222222 │
 │           ┆ 0.953125, ┆           ┆ false,    ┆   ┆ 1, 1, 1,  ┆ true,     ┆ true,     ┆ 22222222 │
 │           ┆ 0.9375,   ┆           ┆ false,    ┆   ┆ 1, 1, 1,  

In [13]:
df.columns

['method',
 'model_version',
 'prompt_version',
 'reasoning',
 'prior_correction',
 'data_path',
 'split',
 'setup_id',
 'n_boards',
 'correct_square',
 'correct_square_mean',
 'correct_board',
 'correct_board_mean',
 'board_rank',
 'board_rank_mean',
 'first_output_illegal',
 'first_output_illegal_mean',
 'none_legal',
 'none_legal_mean',
 'input_tokens',
 'input_tokens_mean',
 'output_tokens',
 'output_tokens_mean',
 'inference_time',
 'inference_time_mean',
 'cost',
 'cost_mean',
 'n_suggested',
 'n_suggested_mean',
 'effort']

In [14]:
df["method"].unique()

method
str
"""cnn"""
"""square_logits"""
"""board"""
"""fen_whole"""
"""square_label"""
"""move"""


In [15]:
(
    df.filter(pl.col("split").eq("test"), pl.col("model_version").str.contains("claude"))
    .group_by("method", "model_version", "reasoning", "effort")
    .agg(
        pl.col("correct_board_mean").mean()
    )
    .sort("correct_board_mean", descending=True)
)

method,model_version,reasoning,effort,correct_board_mean
str,str,str,str,f64
"""square_logits""","""claude-opus-5""","""thinking""","""low""",0.226659


In [16]:
(
    df.filter(pl.col("method").eq("square_logits"))
    .select("n_boards", "correct_square_mean", "correct_board_mean", "board_rank", "model_version")
)

n_boards,correct_square_mean,correct_board_mean,board_rank,model_version
i64,f64,f64,str,str
18,0.228299,0.0,"""[0.6666666666666667, 0.8, 0.71…","""claude-sonnet-5"""
18,0.215278,0.111111,"""[0.7142857142857143, 0.65, 1.0…","""claude-sonnet-5"""
17,0.205882,0.0,"""[0.8666666666666667, 0.6428571…","""claude-sonnet-5"""
11,0.268466,0.090909,"""[0.32352941176470584, 0.555555…","""claude-sonnet-5"""
10,0.2515625,0.1,"""[0.45238095238095233, 0.354838…","""claude-sonnet-5"""
18,0.159722,0.333333,"""[0.5384615384615384, 1.0, null…","""claude-sonnet-5"""
17,0.215074,0.058824,"""[0.9333333333333333, 1.0, 0.71…","""claude-sonnet-5"""
11,0.296875,0.090909,"""[0.4117647058823529, 0.3888888…","""claude-sonnet-5"""
10,0.2671875,0.1,"""[0.26190476190476186, 0.451612…","""claude-sonnet-5"""


In [17]:
df.filter(pl.col("method").eq("move"))

method,model_version,prompt_version,reasoning,prior_correction,data_path,split,setup_id,n_boards,correct_square,correct_square_mean,correct_board,correct_board_mean,board_rank,board_rank_mean,first_output_illegal,first_output_illegal_mean,none_legal,none_legal_mean,input_tokens,input_tokens_mean,output_tokens,output_tokens_mean,inference_time,inference_time_mean,cost,cost_mean,n_suggested,n_suggested_mean,effort
str,str,i64,str,bool,str,str,str,i64,str,f64,str,f64,str,f64,str,str,str,str,str,f64,str,f64,str,f64,str,f64,str,str,str
"""move""","""claude-opus-4-8""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-09_205328""",18,"""[null, null, null, null, null,…",null,"""[true, true, false, false, fal…",0.111111,"""[null, 1.0, null, null, null, …",0.625,"""[true, false, false, true, fal…","""0.4444444444444444""","""[false, false, false, false, f…","""0.2777777777777778""","""[998, 996, 997, 998, 1000, 100…",1003.444444,"""[19, 24, 34, 27, 39, 29, 45, 4…",48.111111,"""[4.125, 2.5470000000000255, 2.…",4.164111,"""[0.005465, 0.00558, 0.00583500…",0.00622,null,null,"""high"""
"""move""","""claude-opus-4-8""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-22_200019""",17,"""[null, null, null, null, null,…",null,"""[true, false, false, false, fa…",0.058824,"""[1.0, null, null, null, null, …",1.0,"""[false, true, false, true, tru…","""0.5294117647058824""","""[false, true, false, false, tr…","""0.47058823529411764""","""[1047, 1050, 1054, 1056, 1057,…",1056.882353,"""[59, 59, 59, 59, 54, 49, 59, 5…",60.823529,"""[4.234000000000151, 3.25, 2.90…",4.151706,"""[0.00671, 0.006725, 0.006745, …",0.006805,null,null,"""high"""
"""move""","""claude-opus-4-8""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-22_201154""",11,"""[null, null, null, null, null,…",null,"""[false, false, false, false, f…",0.0,"""[null, 0.33333333333333337, nu…",0.333333,"""[false, false, false, false, t…","""0.45454545454545453""","""[false, false, false, false, f…","""0.18181818181818182""","""[1012, 1012, 1010, 1010, 1011,…",1009.090909,"""[49, 49, 59, 49, 59, 59, 59, 5…",55.727273,"""[2.7029999999999745, 3.0309999…",3.957273,"""[0.006285000000000001, 0.00628…",0.006439,null,null,"""high"""
"""move""","""claude-opus-4-8""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-22_203121""",10,"""[null, null, null, null, null,…",null,"""[false, false, false, false, f…",0.0,"""[null, null, null, null, null,…",null,"""[true, false, false, false, tr…","""0.5""","""[true, false, false, false, tr…","""0.5""","""[1022, 1020, 1019, 1020, 1020,…",1019.8,"""[59, 59, 64, 104, 74, 59, 84, …",72.5,"""[3.1569999999999254, 3.75, 6.6…",3.9267,"""[0.006585000000000001, 0.00657…",0.006912,null,null,"""high"""
"""move""","""claude-opus-4-8""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-22_203700""",18,"""[null, null, null, null, null,…",null,"""[false, false, false, false, t…",0.055556,"""[null, null, null, null, 1.0, …",1.0,"""[false, true, true, false, fal…","""0.5555555555555556""","""[false, true, true, false, fal…","""0.5555555555555556""","""[999, 999, 998, 998, 1000, 997…",996.555556,"""[59, 64, 54, 39, 69, 54, 49, 5…",60.888889,"""[2.969000000000051, 3.75, 3.85…",3.633667,"""[0.00647, 0.006595, 0.00634, 0…",0.006505,null,null,"""high"""
"""move""","""claude-sonnet-5""",1,"""none""",false,"""data/generated/data.csv""","""val""","""2026-07-09_205328""",18,"""[null, null, null, null, null,…",null,"""[false, true, false, false, fa…",0.055556,"""[null, null, null, null, null,…",null,"""[true, false, false, true, tru…","""0.8333333333333334""","""[true, false, false, true, tru…","""0.8333333333333334""","""[1173, 1171, 1172, 1173, 1175,…",1178.444444,"""[1024, 182, 159, 1024, 1024, 1…",893.222222,"""[22.219000000011874, 3.7030000…",11.844556,"""[0.018879, 0.006243, 0.005901,…",0.016934,"""[0, 1, 1, 0, 0, 0, 1, 0, 0, 0,…","""0.16666666666666666""","""high"""
"""move""","""claud

In [18]:
df.group_by(["method", "model_version"]).agg(pl.col("correct_board_mean").mean(), pl.col("none_legal").mean(), pl.len().alias("n_setups"))

method,model_version,correct_board_mean,none_legal,n_setups
str,str,f64,str,u32
"""square_label""","""claude-sonnet-5""",0.128705,null,10
"""cnn""","""none_global""",0.478962,null,16
"""cnn""","""optimised_plus_prior_correctio…",0.946495,null,16
"""cnn""","""optimised""",0.945928,null,16
"""cnn""","""square_global""",0.913824,null,16
"""cnn""","""square_per_square""",0.894905,null,16
"""fen_whole""","""claude-sonnet-5""",0.0,null,5
"""board""","""claude-sonnet-5""",0.03366,null,10
"""square_logits""","""claude-sonnet-5""",0.105175,null,10


In [19]:
import os
os.getcwd()

'c:\\Users\\User\\Documents\\Coding\\robot-chess-commentator\\evaluation'

## Hypothesis tests and confidence intervals

In [20]:
# 1. Identify the strongest CNN-based approach, and the strongest Claude-based approach
df.head()
cnn = (
    df.filter(pl.col("method").eq("cnn"), pl.col("split").eq("val"))
    .group_by("model_version", "prompt_version", "prior_correction")
    .agg(pl.col("correct_board_mean").mean())
    .sort("correct_board_mean", descending=True)
    .head(1)
)
claude = (
    df.filter(pl.col("method").ne("cnn"), pl.col("split").eq("val"))
    .group_by("method", "model_version", "prompt_version", "reasoning", "effort")
    .agg(pl.col("correct_board_mean").mean())
    .sort("correct_board_mean", descending=True)
    .head(1)
)
cnn, claude

(shape: (1, 4)
 ┌─────────────────────────────────┬────────────────┬──────────────────┬────────────────────┐
 │ model_version                   ┆ prompt_version ┆ prior_correction ┆ correct_board_mean │
 │ ---                             ┆ ---            ┆ ---              ┆ ---                │
 │ str                             ┆ i64            ┆ bool             ┆ f64                │
 ╞═════════════════════════════════╪════════════════╪══════════════════╪════════════════════╡
 │ optimised_plus_prior_correctio… ┆ 1              ┆ true             ┆ 0.947831           │
 └─────────────────────────────────┴────────────────┴──────────────────┴────────────────────┘,
 shape: (1, 6)
 ┌───────────────┬───────────────┬────────────────┬───────────┬────────┬────────────────────┐
 │ method        ┆ model_version ┆ prompt_version ┆ reasoning ┆ effort ┆ correct_board_mean │
 │ ---           ┆ ---           ┆ ---            ┆ ---       ┆ ---    ┆ ---                │
 │ str           ┆ str       

In [21]:
cnn_rows = df.filter(pl.col("model_version").eq("optimised_plus_prior_correction"), pl.col("split").eq("test"))
claude_rows = df.filter(pl.col("model_version").eq("claude-opus-5"), pl.col("method").eq("square_logits"), pl.col("effort").eq("low"), pl.col("split").eq("test"))
cnn_rows, claude_rows

(shape: (11, 30)
 ┌────────┬─────────────┬────────────┬───────────┬───┬───────────┬────────────┬────────────┬────────┐
 │ method ┆ model_versi ┆ prompt_ver ┆ reasoning ┆ … ┆ cost_mean ┆ n_suggeste ┆ n_suggeste ┆ effort │
 │ ---    ┆ on          ┆ sion       ┆ ---       ┆   ┆ ---       ┆ d          ┆ d_mean     ┆ ---    │
 │ str    ┆ ---         ┆ ---        ┆ str       ┆   ┆ f64       ┆ ---        ┆ ---        ┆ str    │
 │        ┆ str         ┆ i64        ┆           ┆   ┆           ┆ str        ┆ str        ┆        │
 ╞════════╪═════════════╪════════════╪═══════════╪═══╪═══════════╪════════════╪════════════╪════════╡
 │ cnn    ┆ optimised_p ┆ 1          ┆ none      ┆ … ┆ 0.0       ┆ null       ┆ null       ┆ high   │
 │        ┆ lus_prior_c ┆            ┆           ┆   ┆           ┆            ┆            ┆        │
 │        ┆ orrectio…   ┆            ┆           ┆   ┆           ┆            ┆            ┆        │
 │ cnn    ┆ optimised_p ┆ 1          ┆ none      ┆ … ┆ 0.0       

In [22]:
cnn_rows["correct_board_mean"].mean(), claude_rows["correct_board_mean"].mean()

(0.9458874458874459, 0.22665945165945162)

In [23]:
import numpy as np
def bootstrap_ci(values: np.ndarray, B=10000, alpha=0.05, method="percentile"):
    point_estimate = values.mean()

    # 10000 times: subsample new arrays
    samples = np.random.choice(values, size=(B, values.size))

    estimates = samples.mean(axis=1)
    lower_q, upper_q = np.quantile(estimates, q=[alpha/2, 1 - alpha/2])

    if method != "percentile":
        return 2 * point_estimate - upper_q, 2 * point_estimate - lower_q
    return lower_q, upper_q

In [24]:
# Reason for percentile bootstrap interval: guaranteed to remain between 0 and 1
cnn_accuracies = cnn_rows["correct_board_mean"].to_numpy()
claude_accuracies = claude_rows["correct_board_mean"].to_numpy()
print(bootstrap_ci(cnn_accuracies))
print(bootstrap_ci(claude_accuracies))

(np.float64(0.8830627705627704), np.float64(1.0))
(np.float64(0.13383838383838384), np.float64(0.31919913419913415))


In [25]:
# Hypothesis test
n_cnn_better = 0
results = {}
for setup in cnn_rows["setup_id"].to_list():
    results[setup] = {
        "cnn_acc": cnn_rows.filter(pl.col("setup_id").eq(setup))["correct_board_mean"].item(),
        "claude_acc": claude_rows.filter(pl.col("setup_id").eq(setup))["correct_board_mean"].item()
    }
    results[setup]["d"] = results[setup]["cnn_acc"] - results[setup]["claude_acc"]
    if results[setup]["d"] > 0:
        n_cnn_better += 1

test_stat = n_cnn_better / len(results)

In [26]:
# p-value
# reference distrbution 

In [27]:
import scipy.stats as stats

n = len(results)
as_extreme = []

for i in range(n + 1):
    if np.abs(i / n - 0.5) >= np.abs(test_stat - 0.5):
        as_extreme.append(i) 

p_val = stats.Binomial(n=n, p=0.5).pmf(as_extreme).sum()
p_val
#stats.Binomial(n=11, p=0.5).pmf([0, 1])


np.float64(0.0009765625)

In [28]:
# Plot distribution of test statistic
proportions = []
probs = []
bin = stats.Binomial(n=n, p=0.5)
for i in range(n + 1):
    proportions.append(i / n)
    probs.append(bin.pmf(i))

pl.DataFrame({"test_stat": proportions, "prob": probs}).write_csv("test_distribution.csv")


In [29]:
2 / 2**11

0.0009765625